In [4]:
%config Completer.use_jedi = False
%matplotlib qt

import json
import numpy as np
import random
import networkx as nx
import pyqtgraph as pg

from pyqtgraph.Qt import QtWidgets, QtCore, QtGui
from networkx.readwrite import json_graph

def arg_vec(u, v):
    rcos = np.sum(u * v)
    rsin = u[0] * v[1] - u[1] * v[0]
    return np.arctan2(rsin, rcos)

def minimum_image_displacement(obj, p1, p2): # periodic boundary
    disp = p2 - p1

    disp9 = np.vstack([
        disp + shift
        for shift in obj.shifts
    ])

    dist_sq9 = np.sum(disp9 ** 2, axis=1)

    k = np.argmin(dist_sq9)

    return disp9[k], k


def build_base_graph(obj):

    normal_x = []
    normal_y = []

    periodic_x = []
    periodic_y = []

    obj.edge_geometry = [] # edgeごとの中央セル上のgeometry

    for edge_id, (i, j) in enumerate(obj.edges):
        p1 = obj.nodes[i]
        p2 = obj.nodes[j]

        disp, k = minimum_image_displacement(
            obj,
            p1,
            p2
        )

        p2_image = p1 + disp

        x = [p1[0], p2_image[0]]
        y = [p1[1], p2_image[1]]
        
        obj.edge_geometry.append({ # edge geometryを保存
            "edge_id": edge_id,
            "edge": (i, j),
            "p1": p1.copy(),
            "p2": p2_image.copy(),
            "periodic": (k != 4)
        })

        if k == 4: # 通常edge
            normal_x.extend(x + [np.nan])
            normal_y.extend(y + [np.nan])

        else: # 周期境界を跨ぐedge
            periodic_x.extend(x + [np.nan])
            periodic_y.extend(y + [np.nan])

    return (
        np.array(normal_x),
        np.array(normal_y),
        np.array(periodic_x),
        np.array(periodic_y)
    )

def update_view(obj): # display repeated copies depending on displayed ranges
    print("update_view called")
    print("viewRange:", obj.plot.viewRange())
    view_range = obj.plot.viewRange()

    xmin, xmax = view_range[0]
    ymin, ymax = view_range[1]

    ix_min = int(np.floor(xmin / obj.L)) - 1
    ix_max = int(np.floor(xmax / obj.L)) + 1

    iy_min = int(np.floor(ymin / obj.L)) - 1
    iy_max = int(np.floor(ymax / obj.L)) + 1

    normal_x = []
    normal_y = []

    periodic_x = []
    periodic_y = []

    node_x = []
    node_y = []

    for ix in range(ix_min, ix_max + 1):

        for iy in range(iy_min, iy_max + 1):

            shift_x = ix * obj.L
            shift_y = iy * obj.L

            normal_x.append(
                obj.base_normal_x + shift_x
            )

            normal_y.append(
                obj.base_normal_y + shift_y
            )

            periodic_x.append(
                obj.base_periodic_x + shift_x
            )

            periodic_y.append(
                obj.base_periodic_y + shift_y
            )

            node_x.append(
                obj.nodes[:, 0] + shift_x
            )

            node_y.append(
                obj.nodes[:, 1] + shift_y
            )

    normal_x = np.concatenate(normal_x)
    normal_y = np.concatenate(normal_y)

    periodic_x = np.concatenate(periodic_x)
    periodic_y = np.concatenate(periodic_y)

    node_x = np.concatenate(node_x)
    node_y = np.concatenate(node_y)

    obj.edge_item.setData(
        normal_x,
        normal_y
    )

    obj.edge_periodic_item.setData(
        periodic_x,
        periodic_y
    )

    obj.node_item.setData(
        x=node_x,
        y=node_y
    )

    box_x = [] # boundary of repeated unit
    box_y = []

    for ix in range(ix_min, ix_max + 1):

        for iy in range(iy_min, iy_max + 1):

            x0 = ix * obj.L
            y0 = iy * obj.L

            # horizontal
            box_x.extend([
                x0, x0 + obj.L,
                np.nan,
                x0, x0 + obj.L,
                np.nan
            ])

            box_y.extend([
                y0, y0,
                np.nan,
                y0 + obj.L, y0 + obj.L,
                np.nan
            ])

            # vertical
            box_x.extend([
                x0, x0,
                np.nan,
                x0 + obj.L, x0 + obj.L,
                np.nan
            ])

            box_y.extend([
                y0, y0 + obj.L,
                np.nan,
                y0, y0 + obj.L,
                np.nan
            ])

    obj.box_item.setData(
        np.array(box_x),
        np.array(box_y)
    )

def kukei(ax, L):
    ax.plot([0, 0], [0, L], lw = 0.5, color='0.0', linestyle='--')
    ax.plot([0, L], [0, 0], lw = 0.5, color='0.0', linestyle='--')
    ax.plot([0, L], [L, L], lw = 0.5, color='0.0', linestyle='--')
    ax.plot([L, L], [0, L], lw = 0.5, color='0.0', linestyle='--')


class Periodic:
    def __init__(self, L, fig_frame, plot_styles=None):
        self.L = L
        self.shifts = np.array([
            [-L, -L], [0, -L], [L, -L],
            [-L,  0], [0,  0], [L,  0],
            [-L,  L], [0,  L], [L,  L],
        ])
        self.fig_frame = fig_frame

        self.window = QtWidgets.QWidget()
        self.styles = {}  # Initialize with an empty dictionary

        default_styles = {
            "edge": {"color": (90, 38, 32), "width": 6.0, "style": QtCore.Qt.PenStyle.SolidLine},
            "edge_periodic": {"color": (90, 38, 32), "width": 3.0, "style": QtCore.Qt.PenStyle.SolidLine},
            "node": {"size": 12, "color": (90, 38, 32)},
            "box": {"color": (128, 128, 128), "width": 1.5, "style": QtCore.Qt.PenStyle.DashLine}
        }
         
        self.styles = default_styles # 2. ユーザー指定の設定があれば、デフォルト設定を上書きする
        if plot_styles:
            for key, value in plot_styles.items(): # ネスト（二重）された辞書を安全に上書き更新する
                if key in self.styles and isinstance(value, dict):
                    self.styles[key].update(value)

        self.score_label = QtWidgets.QLabel("Score: --") # Score board: von Neumann entropy
        self.score_label.setAlignment(QtCore.Qt.AlignmentFlag.AlignCenter)
        self.score_label.setStyleSheet("""
            color: rgb(90, 38, 32); 
            background-color: rgb(248, 246, 235); 
            font-size: 36px; 
            font-weight: 500;
            padding: 6px 12px;
            border-bottom: none;
        """)
        self.score_label.setMinimumHeight(30)
        self.score_label.setVisible(True)
        
        self.message_text_edit = QtWidgets.QTextEdit()
        self.message_text_edit.setStyleSheet("color: white;")  # Set text color and background
        self.message_text_edit.setReadOnly(True)  # Make it read-only
        self.message_text_edit.setAlignment(QtCore.Qt.AlignmentFlag.AlignTop)  # Align text to the top
        self.message_text_edit.setVerticalScrollBarPolicy(QtCore.Qt.ScrollBarPolicy.ScrollBarAlwaysOn)  # Set vertical scroll bar policy
        initial_message = "Start Kagomekagome"
        initial_message += "\n" + "Click an edge to select it (it will turn pink). Press [Diagonal Flip] to flip the selected edge."
        self.display_message(initial_message)

        self.layout = QtWidgets.QVBoxLayout(self.window)  # Initialize the layout with the QWidget
        self.layout.addWidget(self.score_label)
        self.layout.addWidget(self.message_text_edit)
        self.score_label.setMinimumHeight(30)
        self.message_text_edit.setMaximumHeight(50)
        
        self.selected_edge = None
        self.selected_edge_geometry = None
            
        self.hover_edge = None
        self.hover_edge_geometry = None

        print("score_label parent:", self.score_label.parent())
        print("window layout:", self.window.layout())
        print("score_label visible:", self.score_label.isVisible())
        print("window visible:", self.window.isVisible())
        print("score visible:", self.score_label.isVisible())
        print("message visible:", self.message_text_edit.isVisible())

        self.calculate_km_reference()
        self.best_score = None
        self.best_score_step = None

        #self.message_text_edit = QtWidgets.QTextEdit()
        #self.message_text_edit.setStyleSheet("color: white; background-color: black;")  # Set text color and background
        #self.message_text_edit.setReadOnly(True)  # Make it read-only to prevent user editing
        #self.message_text_edit.setAlignment(QtCore.Qt.AlignmentFlag.AlignTop)  # Align text to the top

        #self.message_text_edit.setVerticalScrollBarPolicy(QtCore.Qt.ScrollBarPolicy.ScrollBarAlwaysOn) # Set vertical scroll bar policy
        #layout = QtWidgets.QVBoxLayout(fig_frame)  # Adding the QTextEdit to your layout (assuming you have a layout)
        #layout.addWidget(self.message_text_edit)

    def display_message(self, message):
        # Update the QTextEdit to display the latest message only
        self.message_text_edit.clear()  # Clear previous messages
        message = message.replace("\n", "<br>")
        try:
            message += f"<br>Best score so far: {self.best_score} at step {self.best_score_step}"
        except AttributeError:
            pass
        self.message_text_edit.insertHtml(f"<span style='color: white;'>{message}</span>")  # Insert the latest message

    def score_km_wasserstein(self):
        distance = self.calculate_spectral_distance()
        
        if self.initial_distance == 0:
            return 0.0
        
        score = distance / self.initial_distance
        #score = (score - 0.0)/(1 - 0.0)
        score = - 100 * np.log10(score)
        
        return max(score, 1e-12)
        
    def update_score(self, score):
        self.score_label.setText(
            f"Score: {score:,.0f}"
        )
    
    def load_json(self, fname, magnify):
        self.fname = fname
        with open(self.fname + ".json", "r") as f:
            data = json.load(f)
        
        H = json_graph.node_link_graph(data)
        nodes_load = H.nodes(data=True)
        edges_load = H.edges()
        nodes = {}
        edges = {}
        self.nodes = magnify * np.array([[value["x"], value["y"]] for _, value in nodes_load]) % self.L
        self.edges = [(int(u), int(v)) for u, v in edges_load]
        print("loaded nodes:", self.nodes.shape)
        print("loaded edges:", len(self.edges))

        self.initial_edges = self.edges.copy()
        self.move_history = []
        self.current_step = 0

        print(f"{self.fname} has been loaded and magnified by {magnify} times.")
        self.initial_distance = self.calculate_spectral_distance()
        score = self.score_km_wasserstein()
        entropy_shannon, cycle_counts = self.calculate_cycle_entropy()
        print("Entropy:", entropy_shannon)
        print("Cycle counts:", cycle_counts)
        self.update_score(1e6 * entropy_shannon)
        self.best_score = 1e6 * entropy_shannon
        self.best_score_step = self.current_step

    def calculate_km_reference(self, n_grid=10000):
        """Kesten–McKay distribution for 3-regular graphs."""
        x = np.linspace(-2 * np.sqrt(2), 2 * np.sqrt(2), n_grid)
        rho = (
            3 * np.sqrt(np.maximum(8 - x**2, 0))
            / (2 * np.pi * (9 - x**2))
        )
        
        dx = x[1] - x[0]
        
        cdf = np.zeros_like(x)
        cdf[1:] = np.cumsum(
            0.5 * (rho[:-1] + rho[1:]) * dx
        )
        
        cdf /= cdf[-1]
        
        self.km_x = x
        self.km_cdf = cdf
    
    def set_reference_spectrum(self):
        """現在のグラフのスペクトルを基準として保存する"""
        self.reference_spectrum = self.calculate_laplacian_eigenvalues()
        
    def calculate_von_neumann_entropy(self):
        N = len(self.nodes)
        A = np.zeros((N, N), dtype=float)
        
        for u, v in self.edges:
            A[u, v] = 1.0
            A[v, u] = 1.0
        
        L = 3.0 * np.eye(N) - A
        
        eigenvalues = np.linalg.eigvalsh(L)
        eigenvalues = np.clip(eigenvalues, 0.0, None)
        
        p = eigenvalues / eigenvalues.sum()
        p = p[p > 0]
        
        entropy = -np.sum(p * np.log(p))
        
        return entropy

    def calculate_laplacian_eigenvalues(self):
        N = len(self.nodes)
        A = np.zeros((N, N), dtype=float)
        
        for u, v in self.edges:
            A[u, v] = 1.0
            A[v, u] = 1.0
        
        L = 3.0 * np.eye(N) - A
        
        return np.linalg.eigvalsh(L)

    def calculate_spectral_distance(self):
        """Wasserstein-1 distance from the Kesten–McKay distribution."""
        N = len(self.nodes)
        A = np.zeros((N, N), dtype=float)
        
        for u, v in self.edges:
            A[u, v] = 1.0
            A[v, u] = 1.0
        
        eigenvalues = np.linalg.eigvalsh(A)
        
        p = (np.arange(N) + 0.5) / N # Empirical spectral quantiles
        km_quantiles = np.interp(p, self.km_cdf, self.km_x) # Kesten–McKay quantiles
        
        return np.mean(np.abs(
            np.sort(eigenvalues) - km_quantiles
        ))

    def calculate_cycle_entropy(self, max_len=24):
        from collections import deque
        N = len(self.nodes)

        adj = [set() for _ in range(N)]
        for u, v in self.edges:
            if u != v:
                adj[u].add(v)
                adj[v].add(u)

        cycle_lengths = []

        print("N:", N)
        print("Edges:", len(self.edges))
        print("Degree distribution:", np.unique(
            [len(a) for a in adj], return_counts=True
        ))

        for node in range(N):
            neighbors = sorted(adj[node])
            min_cycle = None

            # Examine every pair of neighbours
            for i, a in enumerate(neighbors):
                for b in neighbors[i + 1:]:

                    # BFS from a to b, excluding the original node
                    queue = deque([(a, 0)])
                    visited = {a}

                    while queue:
                        current, distance = queue.popleft()

                        # Close the cycle
                        if current == b:
                            cycle_length = distance + 2

                            if min_cycle is None or cycle_length < min_cycle:
                                min_cycle = cycle_length

                            break

                        # Maximum cycle length is max_len
                        if distance >= max_len - 2:
                            continue

                        for nxt in adj[current]:
                            if nxt == node or nxt in visited:
                                continue

                            visited.add(nxt)
                            queue.append((nxt, distance + 1))

            cycle_lengths.append(min_cycle)

        # Count the minimum cycle lengths
        cycle_counts = {}
        for length in cycle_lengths:
            if length is not None:
                cycle_counts[length] = cycle_counts.get(length, 0) + 1

        # Calculate Shannon entropy
        total = sum(cycle_counts.values())
        print("cycle_counts.values()", cycle_counts.values())

        if total == 0:
            entropy = 0.0
        else:
            p = np.array(list(cycle_counts.values()), dtype=float) / total
            entropy = -np.sum(p * np.log(p))

        unclassified = cycle_lengths.count(None)

        print(f"Nodes without a cycle of length <= {max_len}: {unclassified}")

        return entropy, dict(sorted(cycle_counts.items()))

    
    def point_to_segment_distance(self, point, p1, p2):
        v = p2 - p1
        vv = np.dot(v, v)
        
        if vv == 0:
            return np.linalg.norm(point - p1)
        
        t = np.dot(point - p1, v) / vv
        t = np.clip(t, 0.0, 1.0)
        
        closest = p1 + t * v
        
        return np.linalg.norm(point - closest)

    def find_nearest_edge(self, pos):
        x = pos.x()
        y = pos.y()
        
        point = np.array([x, y])
        
        ix = int(np.floor(x / self.L)) # カーソルが存在する周期セル
        iy = int(np.floor(y / self.L))
        
        best_distance = np.inf # 現在のセル周辺だけ調べる # 全周期コピーを調べないので軽い
        best_edge_id = None
        best_p1 = None
        best_p2 = None
        
        for dx in (-1, 0, 1):
            for dy in (-1, 0, 1):
                shift = np.array([
                    (ix + dx) * self.L,
                    (iy + dy) * self.L       
                ])
                
                for geometry in self.edge_geometry:
                    p1 = geometry["p1"] + shift
                    p2 = geometry["p2"] + shift
                    
                    distance = self.point_to_segment_distance(
                        point,
                        p1,
                        p2
                    )
                    
                    if distance < best_distance:
                        best_distance = distance
                        best_edge_id = geometry["edge_id"]
                        best_p1 = p1.copy()
                        best_p2 = p2.copy()
        
        view_range = self.plot.viewRange() # クリック／hover判定の距離 # 画面上で一定の太さになるようにviewRangeから決める
        xrange = view_range[0][1] - view_range[0][0]
        yrange = view_range[1][1] - view_range[1][0]
        
        threshold = max(
            xrange,
            yrange
        ) * 0.015 # 画面幅のおよそ1.5%
        
        if best_distance <= threshold:
            return (
                best_edge_id,
                best_p1,
                best_p2
            )
        
        return None

    def update_hover(self, pos):
        result = self.find_nearest_edge(pos)
        
        if result is None: # edge上にいない
            if self.hover_edge is not None:
                self.hover_edge = None
                self.hover_edge_geometry = None
                
                self.hover_edge_item.setData(
                    [],
                    []
                )
            
            return
        
        edge_id, p1, p2 = result
        
        if edge_id == self.selected_edge: # 選択中のedgeならhover色を出さない
            self.hover_edge = None
            self.hover_edge_geometry = None
            
            self.hover_edge_item.setData(
                [],
                []
            )
            
            return
        
        self.hover_edge = edge_id # hoverしているedge
        self.hover_edge_geometry = (
            p1.copy(),
            p2.copy()
        )
        
        self.hover_edge_item.setData(
            [p1[0], p2[0]],
            [p1[1], p2[1]]
        )

    def update_hover(self, pos):
        result = self.find_nearest_edge(pos)
        previous_hover = self.hover_edge # 直前までhoverしていたedge
        
        if result is None: # edge上にいない
            self.hover_edge = None
            self.hover_edge_geometry = None
            
            self.hover_edge_item.setData(
                [],
                []
            )
            
            if (
                previous_hover == self.selected_edge
                and self.selected_edge_geometry is not None
            ): # selected edgeを復活
                p1, p2 = self.selected_edge_geometry
                self.selected_edge_item.setData(
                    [p1[0], p2[0]],
                    [p1[1], p2[1]]
                )
            
            return
        
        edge_id, p1, p2 = result # 新しいedgeをhover
        
        if (
            previous_hover == self.selected_edge
            and edge_id != self.selected_edge
            and self.selected_edge_geometry is not None
        ): # 前のhoverがselected edgeだった場合 # 新しいedgeへ移ったのでselected色を復活
            p1_selected, p2_selected = self.selected_edge_geometry
            self.selected_edge_item.setData(
                [p1_selected[0], p2_selected[0]],
                [p1_selected[1], p2_selected[1]]
            )
            
        self.hover_edge = edge_id # hover情報を更新
        self.hover_edge_geometry = (
            p1.copy(),
            p2.copy()
        )
        
        self.hover_edge_item.setData(
            [p1[0], p2[0]],
            [p1[1], p2[1]]
        ) # hover色
        
        if edge_id == self.selected_edge: # selected edge自身なら # selected色を一時的に隠す
            self.selected_edge_item.setData(
                [],
                []
            )

    def select_edge_at(self, pos):
        result = self.find_nearest_edge(pos)
        print("click result:", result)
        
        if result is None:
            print("No edge selected")
            return
        
        edge_id, p1, p2 = result
        print(
            "clicked edge:",
            edge_id,
            self.edges[edge_id]
        )
        
        if self.selected_edge == edge_id: # 同じedgeをもう一度クリック # → 選択解除
            self.selected_edge = None
            self.selected_edge_geometry = None
            
            self.selected_edge_item.setData(
                [],
                []
            )
            
            return
        
        self.selected_edge = edge_id # 新しいedgeを選択
        self.selected_edge_geometry = (
            p1.copy(),
            p2.copy()
        )
        
        self.selected_edge_item.setData(
            [p1[0], p2[0]],
            [p1[1], p2[1]]
        )
        
        self.hover_edge = None # hoverを消す
        self.hover_edge_geometry = None
        self.hover_edge_item.setData(
            [],
            []
        )
        print(
            "SELECTED:",
            edge_id,
            self.edges[edge_id]
        )

    def mouse_moved(self, event):
        pos = event[0]
        mouse_point = self.plot.vb.mapSceneToView(pos) # scene座標 → plot座標
        self.update_hover(mouse_point)

    def mouse_clicked(self, event):
        if event.button() != QtCore.Qt.MouseButton.LeftButton: # 左クリックだけ
            return
        
        pos = event.scenePos() # scene座標
        mouse_point = self.plot.vb.mapSceneToView(pos) # plot座標へ変換
        print(
            "CLICK:",
            mouse_point.x(),
            mouse_point.y()
        )
        
        self.select_edge_at(mouse_point)

    def flip_selected_edge(self):
        if self.selected_edge is None: # 選択されたedgeがない
            print("No edge is selected.")
            return
        
        edge_id = self.selected_edge
        
        a, b = self.edges[edge_id]
        
        print("selected edge:", edge_id, (a, b))
        
        p_a = self.nodes[a] # edge A の中心
        p_b = self.nodes[b]
        
        disp_ab, _ = minimum_image_displacement(
            self,
            p_a,
            p_b
        )
        
        p_b_image = p_a + disp_ab
        
        vector = p_b_image - p_a
        center = 0.5 * (p_a + p_b_image)
        
        a_neighbors = [] # a, b に接続しているedgeを探す
        b_neighbors = []
        
        for eid, (u, v) in enumerate(self.edges):
            if eid == edge_id: # edge A 自身は除外
                continue
            
            if u == a: # aに接続
                a_neighbors.append((eid, v))
            
            elif v == a:
                a_neighbors.append((eid, u))
            
            if u == b: # bに接続
                b_neighbors.append((eid, v))
            
            elif v == b:
                b_neighbors.append((eid, u))
            
        
        combined_neighbors = [neighbor[1] for neighbor in a_neighbors] + [neighbor[1] for neighbor in b_neighbors]
        if len(combined_neighbors) != len(set(combined_neighbors)):
            message = f"The selected edge that constitutes a triangle cannot be flipped: {combined_neighbors}"
            message += "\n" + "Consider flipping another edge"
            print(message)  # Optional: still print for the console output
            self.display_message(message)  # Update the message label
            return
    
        if len(a_neighbors) != 2: # 3価頂点であることを確認
            print(
                f"Error: node {a} has "
                f"{len(a_neighbors)} other edges."
            )
            return
        
        if len(b_neighbors) != 2:
            print(
                f"Error: node {b} has "
                f"{len(b_neighbors)} other edges."
            )
            return
        
        surrounding = [] # 4つの外側nodeを集める # (edge_id, node, endpoint) # endpoint = "a" or "b"

        for endpoint, neighbours, sign in zip(["a", "b"], [a_neighbors, b_neighbors], [-1, +1]):
            for eid, node in neighbours:
                p = self.nodes[node]
                disp, _ = minimum_image_displacement(
                    self,
                    center,
                    p
                )
                
                position = center + disp
                
                angle = arg_vec(sign * vector, position - center)
                #angle = np.arctan2(
                #    position[1] - center[1],
                #    position[0] - center[0]
                #)
                
                surrounding.append({
                    "edge_id": eid,
                    "node": node,
                    "endpoint": endpoint,
                    "position": position,
                    "angle": angle
                })
            
        surrounding.sort(
            key=lambda x: x["angle"],
            reverse=True
        ) # 時計回りに並べる # atan2は反時計回りなので， # angleの大きい順 = 時計回り
        
        a_items = [
            item
            for item in surrounding
            if item["endpoint"] == "a"
        ] # a1, a2, b1, b2 を決定t # 各endpointについて， # 時計回りに現れる順番を1, 2とする
        
        b_items = [
            item
            for item in surrounding
            if item["endpoint"] == "b"
        ]
        
        a1 = a_items[0]
        a2 = a_items[1]
        
        b1 = b_items[0]
        b2 = b_items[1]

        R = np.array([
            [0, -1],
            [+1, 0]
        ])

        self.nodes[a] = (center + 0.5 * R @ vector) % self.L
        self.nodes[b] = (center - 0.5 * R @ vector) % self.L
        
        print("clockwise order:") # デバッグ表示
        for item in surrounding:
            print(
                item["endpoint"],
                item["node"],
                "edge",
                item["edge_id"],
                "angle",
                item["angle"]
            )
            
        print()
        print("A =", (a, b))
        print("a1 =", a1["node"])
        print("a2 =", a2["node"])
        print("b1 =", b1["node"])
        print("b2 =", b2["node"])
        
        #remove_edges = {
        #    a1["edge_id"],
        #    b1["edge_id"]
        #} # 削除するedge # a2-a # b2-b
        
        #new_edges = [
        #    (a1["node"], b),
        #    (b1["node"], a)
        #] # 新しいedge # a2-b # b2-a
        
        #new_edge_list = [] # self.edgesを更新
        
        #for eid, edge in enumerate(self.edges):
        #    if eid not in remove_edges:
        #        new_edge_list.append(edge)
        
        #new_edge_list.extend(new_edges)
        #self.edges = new_edge_list
        
        removed_edges = [
            self.edges[a1["edge_id"]],
            self.edges[b1["edge_id"]]
        ]
        
        new_edges = [
            (a1["node"], b),
            (b1["node"], a)
        ]

        move_record = {
            "removed_edges": removed_edges,
            "added_edges": new_edges.copy()
        }
        
        for eid, edge in zip(
            [a1["edge_id"], b1["edge_id"]],
            new_edges
        ):
            self.edges[eid] = edge

        self.move_history = self.move_history[:self.current_step] #re do後のflipなら，従来のlogを破棄して更新

        self.move_history.append(move_record)
        self.current_step += 1
        self.update_history_controls()
        print(f"Current step: {self.current_step}")
        
        self.selected_edge = None # 選択状態を解除
        self.selected_edge_geometry = None
            
        self.selected_edge_item.setData(
            [],
            []
        )
        
        self.hover_edge = None
        self.hover_edge_geometry = None
        
        self.hover_edge_item.setData(
            [],
            []
        )
        
        (
            self.base_normal_x,
            self.base_normal_y,
            self.base_periodic_x,
            self.base_periodic_y
        ) = build_base_graph(self) # グラフのgeometryを再構築
        
        update_view(self) # 表示を更新
        
        print()
        print("Graph transformed.")
        print("Removed:")
        print(
            (a, a2["node"]),
            (b, b2["node"])
        )
        
        print("Added:")
        print(
            (a2["node"], b),
            (b2["node"], a)
        )
        #score = self.score_km_wasserstein()
        entropy = self.calculate_von_neumann_entropy()
        entropy_shannon, cycle_counts = self.calculate_cycle_entropy()
        print("entropy_shannon", entropy_shannon)
        message = "Successfully flipped!"
        message += "\n" + f"von Neumann entropy: {entropy}" + f", n-gon Shannon entropy: {entropy_shannon}"
        #message += "   " + f"[step {self.current_step}]"
        self.update_score(1e6 * entropy_shannon)
        self.display_message(message)
        score = 1e6 * entropy_shannon
        if score > self.best_score:
            self.best_score = score
            self.best_score_step = self.current_step
        #self.best_score = max(1e6 * entropy_shannon, self.best_score)

    def random_flip_edge(self):
        if not self.edges:# Choose a random edge
            print("No edges available to flip.")
            return
        
        edge_id = random.choice(range(len(self.edges)))  # Randomly select an edge index
        print(f"Selected edge for random flip: {edge_id}")
        
        self.selected_edge = edge_id # Call the existing flip functionality with the selected edge
        self.flip_selected_edge()

    def undo(self):
        if self.current_step > 0:
            self.recover_graph(
                self.current_step - 1,
                force_current=True
            )
    
    def redo(self):
        if self.current_step < len(self.move_history):
            self.recover_graph(
                self.current_step + 1,
                force_current=True
            )
    
    def go_to_step(self, step):
        if 0 <= step <= len(self.move_history):
            self.recover_graph(
                step,
                force_current=False
            )

    def update_history_controls(self):
        if not hasattr(self, "step_spinbox"):
            return

        max_step = len(self.move_history)
        
        self.step_spinbox.setRange(0, max_step)
        self.step_spinbox.setValue(self.current_step)
        
        self.max_step_label.setText(f"/{max_step}")

    def apply_move(self, edges, move, forward=True):
        before = (
            move["removed_edges"]
            if forward
            else move["added_edges"]
        )
        
        after = (
            move["added_edges"]
            if forward
                else move["removed_edges"]
        )
        
        for old_edge, new_edge in zip(before, after):
            key = frozenset(old_edge)
            
            for eid, edge in enumerate(edges):
                if frozenset(edge) == key:
                    edges[eid] = tuple(new_edge)
                    break
            else:
                raise ValueError(
                    f"Edge {old_edge} not found in graph."
                )
        
        return edges

    def recover_graph(self, target_step, force_current=False):
        if not 0 <= target_step <= len(self.move_history): # ステップ番号の範囲を確認
            print(f"Invalid step: {target_step}")
            return
        
        d_initial = target_step # 復元開始地点を決定
        d_current = abs(target_step - self.current_step)
        
        if force_current or d_current <= d_initial:
            edges = self.edges.copy() # 現在のグラフから復元
            start_step = self.current_step
        
        else: # 初期グラフから復元
            edges = self.initial_edges.copy()
            start_step = 0
        
        if target_step > start_step: # 必要なmoveを順方向に適用
            for move in self.move_history[start_step:target_step]:
                edges = self.apply_move(
                    edges,
                    move,
                    forward=True
                )
        
        elif target_step < start_step: # 必要なmoveを逆方向に適用
            
            for i in range(start_step - 1, target_step - 1, -1):
                edges = self.apply_move(
                    edges,
                    self.move_history[i],
                    forward=False
                )
        
        self.edges = edges # グラフを更新
        self.current_step = target_step
        
        
        self.selected_edge = None # 選択・hover状態を解除
        self.selected_edge_geometry = None
        self.hover_edge = None
        self.hover_edge_geometry = None
        
        self.selected_edge_item.setData([], [])
        self.hover_edge_item.setData([], [])
        
        (
            self.base_normal_x,
            self.base_normal_y,
            self.base_periodic_x,
            self.base_periodic_y
        ) = build_base_graph(self) # 描画geometryを再構築
        
        update_view(self)
        
        entropy = self.calculate_von_neumann_entropy() # スコアを更新
        entropy_shannon, cycle_counts = self.calculate_cycle_entropy()
        
        self.update_score(1e6 * entropy_shannon)
        self.update_history_controls() # ステップ表示を更新
        entropy = self.calculate_von_neumann_entropy()
        entropy_shannon, cycle_counts = self.calculate_cycle_entropy()
        print("entropy_shannon", entropy_shannon)
        message = f"Recovered graph at step {self.current_step}"
        message += "\n" + f"von Neumann entropy: {entropy}" + f", n-gon Shannon entropy: {entropy_shannon}"
        #message += "   " + f"[step {self.current_step}]"
        self.update_score(1e6 * entropy_shannon)
        self.display_message(message)
        self.best_score = max(1e6 * entropy_shannon, self.best_score)
        print(f"Recovered graph at step {self.current_step}")


    def spring_relax(self, time_steps, k_spring=0.01, l_free=None):
        nodes_sim = self.nodes.copy() # 緩和対象のnode座標
        
        if l_free is None: # 自然長 # 指定しなければ，現在の平均edge長の80%を使う
            edge_lengths = []
            
            for i, j in self.edges:
                disp, _ = minimum_image_displacement(
                    self,
                    nodes_sim[i],
                    nodes_sim[j]
                )
                
                edge_lengths.append(
                    np.linalg.norm(disp)
                )
            
            l_free = np.mean(edge_lengths) * 0.8
            
        l_free_sq = l_free ** 2
        
        forces = np.zeros_like(nodes_sim) # 力
        
        for time_step in range(time_steps): # 緩和
            sum_disp = 0.0
            forces.fill(0.0)
            
            for i, j in self.edges: # 全edgeについて力を計算
                p1 = nodes_sim[i]
                p2 = nodes_sim[j]
                
                disp, k = minimum_image_displacement(
                    self,
                    p1,
                    p2
                ) # minimum image
                
                dist_sq = np.sum(disp ** 2)
                
                pforce1 = -k_spring * (
                    l_free_sq / dist_sq - 1
                ) * disp # 元コードと同じばね力
                
                forces[i] += pforce1
                forces[j] -= pforce1
                
                sum_disp += np.sqrt(dist_sq)
            
            nodes_sim += forces # nodeを移動
            nodes_sim %= self.L # 周期境界
            
            if (
                time_step % 100 == 0
                or time_step == time_steps - 1
            ): # 進捗表示
                mean_length = (
                    sum_disp / len(self.edges)
                )
                
                print(
                    time_step,
                    mean_length / l_free - 1
                )
        
        self.display_message("Relaxed")
        return nodes_sim

    def relax_structure(
        self,
        time_steps=100,
        k_spring=0.1,
        l_free=None
    ):
        print("=== Spring relaxation ===")
        
        new_nodes = self.spring_relax(
            time_steps=time_steps,
            k_spring=k_spring,
            l_free=l_free
        )
        
        self.nodes = new_nodes # node座標を更新
        
        (
            self.base_normal_x,
            self.base_normal_y,
            self.base_periodic_x,
            self.base_periodic_y
        ) = build_base_graph(self) # 描画geometryを再構築
        
        update_view(self) # 表示更新
        
        print("=== Relaxation finished ===")

    def key_press_event(self, event):
        if event.key() == QtCore.Qt.Key.Key_F:
            self.flip_selected_edge()

    def save_graph(self):
        filename, _ = QtWidgets.QFileDialog.getSaveFileName(
            self.container,
            "Save Graph",
            "",
            "JSON Files (*.json)"
        )
        
        if not filename:
            return
        
        data = {
            "L": self.L,
            "fig_frame": self.fig_frame,
            "nodes": self.nodes.tolist(),
            "edges": [list(edge) for edge in self.edges],
            "initial_edges": [list(edge) for edge in self.initial_edges],
            "move_history": self.move_history,
            "current_step": self.current_step,
        }
        
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        
        print(f"Saved: {filename}")
    
    
    def load_graph(self):
        filename, _ = QtWidgets.QFileDialog.getOpenFileName(
            self.container,
            "Load Graph",
            "",
            "JSON Files (*.json)"
        )
        
        if not filename:
            return
        
        with open(filename, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        self.L = data["L"]
        self.fig_frame = data["fig_frame"]
        self.nodes = np.array(data["nodes"], dtype=float)
        self.edges = [tuple(edge) for edge in data["edges"]]
        self.initial_edges = [
            tuple(edge) for edge in data["initial_edges"]
        ]
        self.move_history = data["move_history"]
        self.current_step = data["current_step"]
        
        self.recover_graph(self.current_step, force_current=True)
        
        print(f"Loaded: {filename}")
    
    def show_periodic_graph(self):
        self.app = QtWidgets.QApplication.instance()
        
        if self.app is None:
            self.app = QtWidgets.QApplication([])
        
        pg.setConfigOptions(antialias=True)

        self.win = pg.GraphicsLayoutWidget(
            show=False,
            title="Periodic Graph"
        )
        self.win.setBackground((248, 246, 235))
        self.win.resize(800, 850)
        self.win.ci.layout.setContentsMargins(0, 0, 0, 0)
        self.win.ci.layout.setSpacing(0)
        
        self.plot = self.win.addPlot()
        menu = self.plot.getViewBox().menu
        menu.addSeparator()
        save_action = QtGui.QAction("Save", menu)
        save_action.triggered.connect(self.save_graph)
        menu.addAction(save_action)
        load_action = QtGui.QAction("Load", menu)
        load_action.triggered.connect(self.load_graph)
        menu.addAction(load_action)
        
        self.plot.setAspectLocked(True)
        
        palette = QtWidgets.QApplication.palette()
        COLOR_TEXT = palette.color(
            QtGui.QPalette.ColorRole.WindowText
        )
        COLOR_BACKGROUND = palette.color(
            QtGui.QPalette.ColorRole.Window
        )
        
        #self.plot.showAxis("left", False) # 軸を非表示
        #self.plot.showAxis("bottom", False)
        self.plot.hideAxis("left")
        self.plot.hideAxis("bottom")
        self.plot.hideAxis("right")
        self.plot.hideAxis("top")

        item_configs = {}
        item_configs["edge_item"] = {
            "class": pg.PlotDataItem,
            "kwargs": {"pen": pg.mkPen(self.styles["edge"]["color"], width=self.styles["edge"]["width"], style=self.styles["edge"]["style"])}
        }
        item_configs["edge_periodic_item"] = {
            "class": pg.PlotDataItem,
            "kwargs": {"pen": pg.mkPen(self.styles["edge_periodic"]["color"], width=self.styles["edge_periodic"]["width"], style=self.styles["edge_periodic"]["style"])}
        }
        item_configs["node_item"] = {
            "class": pg.ScatterPlotItem,
            "kwargs": {"size": self.styles["node"]["size"], "pen": None, "brush": pg.mkBrush(self.styles["node"]["color"])}
        }
        item_configs["box_item"] = {
            "class": pg.PlotDataItem,
            "kwargs": {"pen": pg.mkPen(self.styles["box"]["color"], width=self.styles["box"]["width"], style=self.styles["box"]["style"])}
        }
        item_configs["hover_edge_item"] = {
            "class": pg.PlotDataItem,
            "kwargs": {"pen": pg.mkPen((96, 124, 131), width=18)}
        }
        item_configs["selected_edge_item"] = {
            "class": pg.PlotDataItem,
            "kwargs": {"pen": pg.mkPen((237, 139, 135), width=36)}
        }
        
        for attr_name, config in item_configs.items():
            item = config["class"](**config["kwargs"])
            setattr(self, attr_name, item)
            self.plot.addItem(item)

        self.edge_item.setZValue(0) # 描画順（Z-order）
        self.edge_periodic_item.setZValue(0)
        self.box_item.setZValue(-1)
        self.node_item.setZValue(10)
        self.hover_edge_item.setZValue(15)
        self.selected_edge_item.setZValue(5)
        
        (
            self.base_normal_x, 
            self.base_normal_y,
            self.base_periodic_x,
            self.base_periodic_y
        ) = build_base_graph(self)

        print("=== DEBUG ===")
        print("nodes:", self.nodes.shape)
        print("edges:", len(self.edges))
        print("base_normal:", len(self.base_normal_x))
        print("base_periodic:", len(self.base_periodic_x))
        print("plot:", self.plot)
        print("items:", self.plot.items)
        
        self.plot.sigRangeChanged.connect(
            lambda *_: update_view(self)
        ) # スクロール・ズーム時に再描画
        
        self.proxy = pg.SignalProxy(# マウス移動
            self.plot.scene().sigMouseMoved,
            rateLimit=60,
            slot=self.mouse_moved
        )

        self.plot.scene().sigMouseClicked.connect(
            self.mouse_clicked
        )

        self.win.keyPressEvent = self.key_press_event
        
        self.plot.setXRange(
            -self.L * self.fig_frame,
            +self.L * (1 + self.fig_frame),
            padding=0
        ) # 初期表示
        
        self.plot.setYRange(
            -self.L * self.fig_frame,
            +self.L * (1 + self.fig_frame),
            padding=0
        )

        self.plot.setMouseEnabled(x=True, y=True)# マウスによる移動・ズームを有効化
        self.plot.setLimits(
            xMin=None, xMax=None,    # 位置の制限はなし（どこまでも移動できる）
            yMin=None, yMax=None,
            minXRange=0.01 * self.L,   # 最大まで拡大したときの画面の幅（ズームイン制限）
            maxXRange=2.5 * self.L, # 最大まで縮小したときの画面の幅（ズームアウト制限）
            minYRange=0.01 * self.L,
            maxYRange=2.5 * self.L
        )
        
        update_view(self)

        container = QtWidgets.QWidget()

        container_layout = QtWidgets.QVBoxLayout(container)
        container_layout.setContentsMargins(0, 0, 0, 0)
        container_layout.setSpacing(1)

        # Add the score board
        container_layout.addWidget(self.score_label)

        # Add the plot to the layout
        container_layout.addWidget(self.win)        

        # Add the message label below the plot
        container_layout.addWidget(self.message_text_edit)
        
        button_layout = QtWidgets.QHBoxLayout() # UI buttons
        
        transform_button = QtWidgets.QPushButton("Diagonal Flip ↻") # Transform
        transform_button.clicked.connect(
            self.flip_selected_edge
        )
        button_layout.addWidget(transform_button)

        random_flip_button = QtWidgets.QPushButton("Random Diagonal Flip ↻")  # Button for random flip
        random_flip_button.clicked.connect(self.random_flip_edge)  # Connect to the functionality
        button_layout.addWidget(random_flip_button)  # Add button to the layout
        
        relax_button = QtWidgets.QPushButton("Relax") # Relax
        relax_button.clicked.connect(
            lambda: self.relax_structure(
                time_steps=100,
                k_spring=0.1
            )
        )
        button_layout.addWidget(relax_button)

        history_layout = QtWidgets.QHBoxLayout() # 2nd button row
        
        undo_button = QtWidgets.QPushButton("Step Back") # Undo
        undo_button.clicked.connect(self.undo)
        history_layout.addWidget(undo_button)
        
        redo_button = QtWidgets.QPushButton("Next Step") # Redo
        redo_button.clicked.connect(self.redo)
        history_layout.addWidget(redo_button)
        
        self.step_spinbox = QtWidgets.QSpinBox() # Step navigation
        self.step_spinbox.setRange(0, len(self.move_history))
        self.step_spinbox.setValue(self.current_step)
        self.step_spinbox.setButtonSymbols(
            QtWidgets.QAbstractSpinBox.ButtonSymbols.NoButtons
        )
        self.step_spinbox.setMinimumWidth(50)
        self.step_spinbox.setAlignment(
            QtCore.Qt.AlignmentFlag.AlignRight
        )
        history_layout.addWidget(self.step_spinbox)

        self.max_step_label = QtWidgets.QLabel(
            f"/{len(self.move_history)}"
        )
        self.max_step_label.setStyleSheet(
            "color: gray;"
        )
        self.step_spinbox.setFixedWidth(50)
        self.max_step_label.setFixedWidth(50)
        history_layout.addWidget(self.max_step_label)
        
        go_button = QtWidgets.QPushButton("Go")
        go_button.clicked.connect(
            lambda: self.go_to_step(self.step_spinbox.value())
        )
        history_layout.addWidget(go_button)

        container_layout.addLayout(button_layout)
        container_layout.addLayout(history_layout)

        self.undo_shortcut = QtGui.QShortcut(
            QtGui.QKeySequence.StandardKey.Undo, container
        ) # Undo: Command + Z
        self.undo_shortcut.activated.connect(self.undo)
        
        self.redo_shortcut = QtGui.QShortcut(
            QtGui.QKeySequence.StandardKey.Redo, container
        ) # Redo: Command + Shift + Z
        self.redo_shortcut.activated.connect(self.redo)
        
        container.resize(800, 900)
        container.show()
        
        self.container = container
        
        self.container.show()
        self.app.exec()
        
        
fig_frame = 0.1

L = 100
#fname = "../graph_20260722_240_360_083719"
#magnify = 4.3
fname = "../graph_20260721_60_90_235627"
magnify = 10

pgraph = Periodic(L, fig_frame)
pgraph.load_json(fname, magnify)
pgraph.show_periodic_graph()

score_label parent: <PySide6.QtWidgets.QWidget(0x600000f5a760) at 0x11631c2c0>
window layout: <PySide6.QtWidgets.QVBoxLayout(0x600003ec3380) at 0x1161ee2c0>
score_label visible: False
window visible: False
score visible: False
message visible: False
loaded nodes: (60, 2)
loaded edges: 90
../graph_20260721_60_90_235627 has been loaded and magnified by 10 times.
N: 60
Edges: 90
Degree distribution: (array([3]), array([60]))
cycle_counts.values() dict_values([60])
Nodes without a cycle of length <= 24: 0
Entropy: -0.0
Cycle counts: {6: 60}
=== DEBUG ===
nodes: (60, 2)
edges: 90
base_normal: 225
base_periodic: 45
plot: <pyqtgraph.graphicsItems.PlotItem.PlotItem.PlotItem(0x600000005b20, parent=0x600000005900, pos=0,0, flags=(ItemUsesExtendedStyleOption|ItemSendsGeometryChanges)) at 0x116334bc0>
items: [<pyqtgraph.graphicsItems.PlotDataItem.PlotDataItem(0x6000008a3110, parent=0x60000089ea90, pos=0,0, flags=(ItemHasNoContents|ItemSendsGeometryChanges)) at 0x156a95f40>, <pyqtgraph.graphicsItem